In [9]:
import os
import json
import time
import random
import psutil
import pandas as pd
from datetime import datetime
from tqdm.notebook import tqdm
from sentence_transformers import util
from IPython.display import display, HTML

# Paths
BASE_DIR = os.path.dirname(os.getcwd())
TEST_SET_PATH = os.path.join(BASE_DIR, "data", "diverse_questions.json")
RESULTS_DIR = os.path.join(BASE_DIR, "data", "eval_results")
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"✅ Setup complete. Results will be saved to: {RESULTS_DIR}")

✅ Setup complete. Results will be saved to: /Users/bhushan/tisd/data/eval_results


In [10]:
import sys
import os
import importlib

# Ensure the notebooks folder is in the path
notebook_dir = os.getcwd()
if notebook_dir not in sys.path:
    sys.path.append(notebook_dir)

# FORCE RELOAD the engine file
import tisd_engine_mlx
importlib.reload(tisd_engine_mlx)
from tisd_engine_mlx import TISDEngine

# Initialize
engine = TISDEngine(verbose=True) # Turn verbose ON for the first load to see paths
engine.load()

print("🚀 v3.3 Engine is online.")

🔍 Engine looking for data in: /Users/bhushan/tisd/vectorstore/chroma_db


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

✅ Engine v3.3 Ready | Collection: tisd_knowledge_base | RAM: 8.62GB
🚀 v3.3 Engine is online.


In [11]:
def get_semantic_score(prediction, ground_truth):
    """Calculates meaning-based similarity using the engine's internal embedder."""
    if not prediction or len(prediction) < 5: return 0.0
    
    # We leverage the MiniLM embedder already loaded in the engine
    emb1 = engine.embedder.encode(prediction, convert_to_tensor=True)
    emb2 = engine.embedder.encode(ground_truth, convert_to_tensor=True)
    
    cosine_score = util.cos_sim(emb1, emb2)
    return round(float(cosine_score), 4)

def get_ram_gb():
    return round(psutil.virtual_memory().used / 1e9, 2)

In [13]:
# Load the Diverse Test Set
if not os.path.exists(TEST_SET_PATH):
    raise FileNotFoundError(f"Missing {TEST_SET_PATH}. Please generate it with Claude first!")

with open(TEST_SET_PATH, 'r') as f:
    full_test_set = json.load(f)

# 1. Randomize the order
random.shuffle(full_test_set)

# 2. Config
TOTAL_QUESTIONS = 100
BATCH_SIZE = 25   
BREAK_DURATION = 45 # seconds
timestamp = datetime.now().strftime("%Y%m%d_%H%M")

current_batch_data = []
part_counter = 1

def cooldown_timer(seconds):
    """A clean in-place countdown timer for the M4 break."""
    for i in range(seconds, 0, -1):
        # \r moves cursor to start of line, end="" prevents newline
        print(f"\r💤 M4 Thermal Recovery: {i:02d}s remaining... ", end="", flush=True)
        time.sleep(1)
    print("\r✅ M4 Silicon Cooled. Resuming Inference...          ")

print(f"🏁 Starting SOTA Evaluation: {TOTAL_QUESTIONS} Questions.")
print(f"🌡️  Protocol: {BREAK_DURATION}s break every {BATCH_SIZE} questions to prevent SSD/SoC heat soak.")

for i in range(TOTAL_QUESTIONS):
    item = full_test_set[i]
    question = item['question']
    ground_truth = item['ground_truth']
    
    # --- INFERENCE ---
    start_inference = time.time()
    # We pass verbose=False to keep the terminal clean
    answer, sources = engine.ask(question, verbose=False)
    latency = round(time.time() - start_inference, 2)
    
    # --- EVALUATION ---
    score = get_semantic_score(answer, ground_truth)
    ram_usage = get_ram_gb()
    
    # Structure the Result entry
    result_entry = {
        "id": i + 1,
        "question": question,
        "ground_truth": ground_truth,
        "tara_answer": answer,
        "retrieved_sources": sources,
        "stats": {
            "semantic_score": score,
            "latency_sec": latency,
            "ram_gb": ram_usage
        }
    }
    
    current_batch_data.append(result_entry)
    print(f"Q{i+1:03}: Score {score:.2f} | Latency {latency}s | RAM {ram_usage}GB")
        
    # --- BATCH SAVE & BREAK LOGIC ---
    # Triggered every 25 questions
    if (i + 1) % BATCH_SIZE == 0:
        # Save the JSON file for this part
        filename = f"Validation_result_part{part_counter}_{timestamp}.json"
        save_path = os.path.join(RESULTS_DIR, filename)
        
        with open(save_path, 'w') as f:
            json.dump(current_batch_data, f, indent=4)
        
        print(f"\n💾 PART {part_counter} SAVED: {filename}")
        
        # Increment part counter and clear current batch
        part_counter += 1
        current_batch_data = []

        # Give the Mac a break (except after the final question)
        if (i + 1) < TOTAL_QUESTIONS:
            cooldown_timer(BREAK_DURATION)

print("\n✨ STRESS TEST COMPLETE. All 4 parts saved to data/eval_results/")

🏁 Starting SOTA Evaluation: 100 Questions.
🌡️  Protocol: 45s break every 25 questions to prevent SSD/SoC heat soak.
Q001: Score 0.89 | Latency 1.89s | RAM 9.86GB
Q002: Score 0.97 | Latency 1.94s | RAM 9.82GB
Q003: Score 0.96 | Latency 1.98s | RAM 9.77GB
Q004: Score 0.80 | Latency 0.59s | RAM 9.81GB
Q005: Score 0.91 | Latency 2.27s | RAM 9.73GB
Q006: Score 0.57 | Latency 1.63s | RAM 9.69GB
Q007: Score 0.79 | Latency 5.89s | RAM 9.87GB
Q008: Score 0.83 | Latency 0.43s | RAM 9.68GB
Q009: Score 0.88 | Latency 0.85s | RAM 9.68GB
Q010: Score 0.79 | Latency 0.53s | RAM 9.67GB
Q011: Score 0.68 | Latency 1.06s | RAM 9.67GB
Q012: Score 0.87 | Latency 2.03s | RAM 9.6GB
Q013: Score 0.88 | Latency 0.45s | RAM 9.63GB
Q014: Score 0.92 | Latency 1.58s | RAM 9.61GB
Q015: Score 0.79 | Latency 2.23s | RAM 9.62GB
Q016: Score 0.84 | Latency 1.49s | RAM 9.61GB
Q017: Score 0.88 | Latency 1.06s | RAM 9.64GB
Q018: Score 0.54 | Latency 2.15s | RAM 9.6GB
Q019: Score 0.76 | Latency 2.08s | RAM 9.61GB
Q020: Score 

In [15]:
# Aggregate all 4 parts for the final report
# We filter by the timestamp generated in Cell 4 to ensure we only look at THIS run
all_files = sorted([f for f in os.listdir(RESULTS_DIR) if timestamp in f])
final_results = []

for f in all_files:
    file_path = os.path.join(RESULTS_DIR, f)
    with open(file_path, 'r') as file:
        final_results.extend(json.load(file))

# Use pandas to flatten the JSON structure
df = pd.json_normalize(final_results)

# FIX: Column names are 'stats.semantic_score', not 'metrics.semantic_score'
# We'll use .get logic to be safe
avg_score = df['stats.semantic_score'].mean()
avg_lat = df['stats.latency_sec'].mean()
max_ram = df['stats.ram_gb'].max()

# --- THE SUMMARY REPORT ---
display(HTML(f"""
<div style="padding:20px; background-color:#1a1a1a; color:#00ff00; border-radius:10px; border:2px solid #00ff00; font-family: monospace;">
    <h2 style="color:#00ff00; margin-top:0;">🏆 TISD STRESS TEST SUMMARY (v3.3)</h2>
    <p style="font-size:18px;">Average Semantic Accuracy: <b>{avg_score:.2%}</b></p>
    <p>Average Response Latency: <b>{avg_lat:.2f} seconds</b></p>
    <p>Peak Memory Usage: <b>{max_ram} GB</b></p>
    <p>Total Samples: 100 | Parts: {len(all_files)}</p>
</div>
"""))

# --- THE FULL DATA VIEW ---
# We enable full row display so you can see all 100 questions
print("\n🔍 FULL EVALUATION LOG (All 100 Questions):")
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_colwidth', 150)

# Create a clean display table
view_df = df[[
    'id', 
    'question', 
    'tara_answer', 
    'ground_truth', 
    'stats.semantic_score', 
    'stats.latency_sec'
]].rename(columns={
    'stats.semantic_score': 'Score',
    'stats.latency_sec': 'Lat(s)',
    'tara_answer': 'Tara',
    'ground_truth': 'Truth'
})

display(view_df.sort_values(by="Score", ascending=False))

# Optional: Export the combined results to a single CSV for your records
csv_filename = f"Full_Evaluation_Combined_{timestamp}.csv"
df.to_csv(os.path.join(RESULTS_DIR, csv_filename), index=False)
print(f"\n💾 Full report exported to {csv_filename}")


🔍 FULL EVALUATION LOG (All 100 Questions):


,id,question,Tara,Truth,Score,Lat(s)
1,2,What is bioluminescence?,"Bioluminescence is the production and emission of light by a living organism as a result of a chemical reaction between luciferin and oxygen, cata...","Bioluminescence is the production and emission of light by living organisms through a biochemical reaction, typically involving the substrate luci...",0.9725,1.94
2,3,What is the difference between weather and climate?,"Weather refers to the short-term atmospheric conditions in a specific place at a specific time, including temperature, humidity, precipitation, an...","Weather refers to short-term atmospheric conditions (temperature, humidity, precipitation, wind) at a specific location and time, varying over hou...",0.9625,1.98
23,24,What is photosynthesis?,"Photosynthesis is the process byelogical process by which green plants, algae, and some bacteria convert light energy, usually from the sun, into ...","Photosynthesis is the biochemical process by which green plants, algae, and cyanobacteria convert light energy (primarily from the sun) into chemi...",0.9566,2.92
35,36,What is the golden ratio?,"The golden ratio, often denoted by the Greek letter phi (φ), is a mathematical constant approximately equal to 1.618033988749894. It is found by d...","The golden ratio (φ, phi) is an irrational mathematical constant approximately equal to 1.6180339887. It is defined such that for two quantities a...",0.9331,2.24
56,57,What is a prime number and what is the largest known prime?,"A prime number is a natural number greater than 1 that has no positive divisors other than 1 and itself. The largest known prime, as of my last up...","A prime number is a natural number greater than 1 that has no positive divisors other than 1 and itself. As of 2024, the largest known prime is 2^...",0.9283,1.92
54,55,What is the Coriolis effect?,"The Coriolis effect is the apparent deflection of the path of an object that moves within a rotating surface. In the context of Earth, it causes m...","The Coriolis effect is an inertial force acting on objects moving within a rotating reference frame. On Earth, it causes freely moving objects to ...",0.9251,1.68
13,14,What causes muscle cramps?,"Muscle cramps can be caused by dehydration, overuse, mineral deficiencies (such as low levels of potassium, calcium, or magnesium), or underlying ...","Muscle cramps are involuntary, sustained contractions of skeletal muscle. Common causes include dehydration, electrolyte imbalances (particularly ...",0.9205,1.58
90,91,What is the difference between accuracy and precision?,"Accuracy refers to how close a measured value is to the true value, while precision refers to how reproducible the measurements are, regardless of...",Accuracy refers to how close a measured value is to the true or accepted value — it measures correctness. Precision refers to how closely repeated...,0.9142,1.09
4,5,What is the tallest mountain on Earth measured from its base?,"The tallest mountain on Earth measured from its base is Mauna Kea in Hawaii. When measured from its sea-level base, it rises 10,203 meters above s...","When measured from its oceanic base, Mauna Kea in Hawaii is the tallest mountain on Earth at approximately 10,210 metres total height, of which 4,...",0.9135,2.27
28,29,What is the function of ATP in cells?,"ATP, or adenosine triphosphate, serves as the primary energy currency in cells. It provides the necessary energy for various cellular processes, i...",Adenosine triphosphate (ATP) is the primary energy currency of all living cells. It consists of adenosine bound to three phosphate groups; hydroly...,0.9082,1.39



💾 Full report exported to Full_Evaluation_Combined_20260403_1759.csv
